In [0]:
with

-- Latest consumption snapshot per deployable account
consumption_latest as (
  select
    a.deployable_account_name

  from main.gtm_gold.account_consumption_daily as a
  where a.horizontal_and_vertical_hierarchy_concatenated_emails like concat('%', :ae_email, '%')
    --and a.deployable_account_name not in ('Gruppo Hera', 'ALIA SERVIZI AMBIENTALI SPA', 'Snam Spa')
    --and a.sales_subregion_level_3 = 'Italy Strategic Core'
    and a.Business_Unit = :business_unit
    and a.sales_subregion_level_1 = :region_level_1
    and a.sales_subregion_level_2 = :region_level_2
    and a.dbu_dollars_t7d_avg   > 0
    and a.fiscal_year >= 2026
    and a.usage_date = (select max(usage_date) from main.gtm_gold.account_consumption_daily)
  group by a.deployable_account_name
),

-- Account IDs for the same filter — used to join with account_product_quarterly
filtered_account_ids as (
  select distinct deployable_account_name, account_id
  from main.gtm_gold.account_consumption_daily
  where horizontal_and_vertical_hierarchy_concatenated_emails like concat('%', :ae_email, '%')
    and Business_Unit = :business_unit
    and sales_subregion_level_1 = :region_level_1
    and sales_subregion_level_2 = :region_level_2
    and usage_date = (select max(usage_date) from main.gtm_gold.account_consumption_daily)
),

-- Product projections with LEAD (CQ+1) and LAG (prev quarter actuals proxy)
account_product_with_next as (
  select
    fa.deployable_account_name,
    apq.account_id,
    apq.product,
    apq.fiscal_year_quarter,
    replace(replace(apq.fiscal_year_quarter, chr(39), ''), ' ', '-') as fiscal_year_quarter_fmt,
    apq.weighted_projection,
    apq.use_case_pipe_count
  from main.gtm_gold.account_product_quarterly apq
  inner join filtered_account_ids fa on apq.account_id = fa.account_id
  where cast(concat('20', substr(apq.fiscal_year_quarter, 4, 2)) as int) >= 2027
),

-- Pivot products to one row per deployable_account_name for the current quarter
pivoted as (
  select
    deployable_account_name,
    fiscal_year_quarter_fmt as fiscal_year_quarter,
    concat(concat('20', substr(fiscal_year_quarter_fmt, 3, 2)), ' ', split(fiscal_year_quarter_fmt, '-')[1]) as fiscal_quarter,
    -- AI
    sum(case when product = 'AI' then coalesce(weighted_projection, 0) else 0 end) as ai_wp,
    sum(case when product = 'AI' then coalesce(use_case_pipe_count, 0) else 0 end) as ai_uc_count,
    -- AI/BI
    sum(case when product = 'AI/BI' then coalesce(weighted_projection, 0) else 0 end) as ai_bi_wp,
    sum(case when product = 'AI/BI' then coalesce(use_case_pipe_count, 0) else 0 end) as ai_bi_uc_count,
    -- DWH
    sum(case when product = 'DWH' then coalesce(weighted_projection, 0) else 0 end) as dwh_wp,
    sum(case when product = 'DWH' then coalesce(use_case_pipe_count, 0) else 0 end) as dwh_uc_count,
    -- FMAPI Partner
    sum(case when product = 'FMAPI Partner' then coalesce(weighted_projection, 0) else 0 end) as fmapi_partner_wp,
    sum(case when product = 'FMAPI Partner' then coalesce(use_case_pipe_count, 0) else 0 end) as fmapi_partner_uc_count,
    -- Lakebase
    sum(case when product = 'Lakebase' then coalesce(weighted_projection, 0) else 0 end) as lakebase_wp,
    sum(case when product = 'Lakebase' then coalesce(use_case_pipe_count, 0) else 0 end) as lakebase_uc_count
    /*
    -- Agent Bricks
    sum(case when product = 'Agent Bricks'          then coalesce(weighted_projection_cq,       0) else 0 end) as agent_bricks_wp,
    sum(case when product = 'Agent Bricks'          then coalesce(weighted_projection_cq_plus1,  0) else 0 end) as agent_bricks_wp_cq_plus1,
    sum(case when product = 'Agent Bricks'          then coalesce(use_case_pipe_count,           0) else 0 end) as agent_bricks_uc_count,
    sum(case when product = 'Agent Bricks'          then coalesce(actuals_cq_minus1,             0) else 0 end) as agent_bricks_actuals_prev_q,
    -- AI Excl FMAPI Partner
    sum(case when product = 'AI Excl FMAPI Partner' then coalesce(weighted_projection_cq,       0) else 0 end) as ai_excl_fmapi_wp,
    sum(case when product = 'AI Excl FMAPI Partner' then coalesce(weighted_projection_cq_plus1,  0) else 0 end) as ai_excl_fmapi_wp_cq_plus1,
    sum(case when product = 'AI Excl FMAPI Partner' then coalesce(use_case_pipe_count,           0) else 0 end) as ai_excl_fmapi_uc_count,
    sum(case when product = 'AI Excl FMAPI Partner' then coalesce(actuals_cq_minus1,             0) else 0 end) as ai_excl_fmapi_actuals_prev_q,
    -- AI Platform
    sum(case when product = 'AI Platform'           then coalesce(weighted_projection_cq,       0) else 0 end) as ai_platform_wp,
    sum(case when product = 'AI Platform'           then coalesce(weighted_projection_cq_plus1,  0) else 0 end) as ai_platform_wp_cq_plus1,
    sum(case when product = 'AI Platform'           then coalesce(use_case_pipe_count,           0) else 0 end) as ai_platform_uc_count,
    sum(case when product = 'AI Platform'           then coalesce(actuals_cq_minus1,             0) else 0 end) as ai_platform_actuals_prev_q,
    -- Lakeflow
    sum(case when product = 'Lakeflow'              then coalesce(weighted_projection_cq,       0) else 0 end) as lakeflow_wp,
    sum(case when product = 'Lakeflow'              then coalesce(weighted_projection_cq_plus1,  0) else 0 end) as lakeflow_wp_cq_plus1,
    sum(case when product = 'Lakeflow'              then coalesce(use_case_pipe_count,           0) else 0 end) as lakeflow_uc_count,
    sum(case when product = 'Lakeflow'              then coalesce(actuals_cq_minus1,             0) else 0 end) as lakeflow_actuals_prev_q,
    -- Lakeflow Connect
    sum(case when product = 'Lakeflow Connect'      then coalesce(weighted_projection_cq,       0) else 0 end) as lakeflow_connect_wp,
    sum(case when product = 'Lakeflow Connect'      then coalesce(weighted_projection_cq_plus1,  0) else 0 end) as lakeflow_connect_wp_cq_plus1,
    sum(case when product = 'Lakeflow Connect'      then coalesce(use_case_pipe_count,           0) else 0 end) as lakeflow_connect_uc_count,
    sum(case when product = 'Lakeflow Connect'      then coalesce(actuals_cq_minus1,             0) else 0 end) as lakeflow_connect_actuals_prev_q,
    -- Lakeflow Pipeline
    sum(case when product = 'Lakeflow Pipeline'     then coalesce(weighted_projection_cq,       0) else 0 end) as lakeflow_pipeline_wp,
    sum(case when product = 'Lakeflow Pipeline'     then coalesce(weighted_projection_cq_plus1,  0) else 0 end) as lakeflow_pipeline_wp_cq_plus1,
    sum(case when product = 'Lakeflow Pipeline'     then coalesce(use_case_pipe_count,           0) else 0 end) as lakeflow_pipeline_uc_count,
    sum(case when product = 'Lakeflow Pipeline'     then coalesce(actuals_cq_minus1,             0) else 0 end) as lakeflow_pipeline_actuals_prev_q,
    -- SAP
    sum(case when product = 'SAP'                   then coalesce(weighted_projection_cq,       0) else 0 end) as sap_wp,
    sum(case when product = 'SAP'                   then coalesce(weighted_projection_cq_plus1,  0) else 0 end) as sap_wp_cq_plus1,
    sum(case when product = 'SAP'                   then coalesce(use_case_pipe_count,           0) else 0 end) as sap_uc_count,
    sum(case when product = 'SAP'                   then coalesce(actuals_cq_minus1,             0) else 0 end) as sap_actuals_prev_q,
    -- Unity Catalog
    sum(case when product = 'Unity Catalog'         then coalesce(weighted_projection_cq,       0) else 0 end) as unity_catalog_wp,
    sum(case when product = 'Unity Catalog'         then coalesce(weighted_projection_cq_plus1,  0) else 0 end) as unity_catalog_wp_cq_plus1,
    sum(case when product = 'Unity Catalog'         then coalesce(use_case_pipe_count,           0) else 0 end) as unity_catalog_uc_count,
    sum(case when product = 'Unity Catalog'         then coalesce(actuals_cq_minus1,             0) else 0 end) as unity_catalog_actuals_prev_q
    */
  from account_product_with_next
  group by deployable_account_name, fiscal_year_quarter_fmt
)

select
  c.deployable_account_name,
  p.fiscal_quarter,
  /* Product projections — current fiscal quarter */
  -- AI
  p.ai_wp,
  p.ai_uc_count,
  -- AI/BI
  p.ai_bi_wp,
  p.ai_bi_uc_count,
  -- DWH
  p.dwh_wp,
  p.dwh_uc_count,
  -- FMAPI Partner
  p.fmapi_partner_wp,
  p.fmapi_partner_uc_count,
  -- Lakebase
  p.lakebase_wp,
  p.lakebase_uc_count,
  /* Current fiscal year flag */
  cast(split(p.fiscal_quarter, ' ')[0] as int) = (
    case when month(current_date()) = 1 then year(current_date()) else year(current_date()) + 1 end
  ) as is_current_fiscal
from consumption_latest c
left join pivoted p on c.deployable_account_name = p.deployable_account_name
order by c.deployable_account_name asc, p.fiscal_quarter asc
